In [4]:
# Retrieving the best configuration
import pandas as pd
from pathlib import Path

REPORT_DIR = Path("../reports")

results_df = pd.read_csv(
    REPORT_DIR / "xgboost_tuning_results.csv"
)

best_result = results_df.sort_values(
    "macro_f1",
    ascending=False
).iloc[0]

metric_columns = {
    "macro_f1",
    "accuracy",
    "training_seconds"
}

best_params = {
    key: best_result[key].item()
    if hasattr(best_result[key], "item")
    else best_result[key]
    for key in best_result.index
    if key not in metric_columns
}

print("Best validation macro F1:", best_result["macro_f1"])
print("Best validation accuracy:", best_result["accuracy"])
print("\nSelected hyperparameters:")
print(best_params)

Best validation macro F1: 0.3940153318293452
Best validation accuracy: 0.6705833333333333

Selected hyperparameters:
{'subsample': 1.0, 'n_estimators': 300.0, 'min_child_weight': 5.0, 'max_depth': 6.0, 'learning_rate': 0.1, 'colsample_bytree': 1.0}


In [5]:
# Check the parameters
print("Best parameters:")
for key, value in best_params.items():
    print(f"{key}: {value} ({type(value).__name__})")

Best parameters:
subsample: 1.0 (float)
n_estimators: 300.0 (float)
min_child_weight: 5.0 (float)
max_depth: 6.0 (float)
learning_rate: 0.1 (float)
colsample_bytree: 1.0 (float)


In [6]:
# Convert parameters to the correct data types
integer_params = [
    "n_estimators",
    "max_depth",
    "min_child_weight",
    "max_leaves",
    "max_bin",
    "num_parallel_tree"
]

for param in integer_params:
    if param in best_params:
        best_params[param] = int(best_params[param])

# Convert other numeric parameters to floats
float_params = [
    "learning_rate",
    "subsample",
    "colsample_bytree",
    "gamma",
    "reg_alpha",
    "reg_lambda"
]

for param in float_params:
    if param in best_params:
        best_params[param] = float(best_params[param])

print("Corrected parameters:", best_params)

Corrected parameters: {'subsample': 1.0, 'n_estimators': 300, 'min_child_weight': 5, 'max_depth': 6, 'learning_rate': 0.1, 'colsample_bytree': 1.0}


In [7]:
import time
import joblib
from pathlib import Path
from xgboost import XGBClassifier

# Initialize the final model
final_xgb = XGBClassifier(
    **best_params,
    objective="multi:softprob",
    eval_metric="mlogloss",
    random_state=42,
    n_jobs=-1
)

# Train the model
start_time = time.time()

final_xgb.fit(X_train, y_train)

training_time = time.time() - start_time

print(f"Training completed in {training_time:.2f} seconds")

# Save the trained model
model_path = Path("../models/final_xgboost_model.joblib")
model_path.parent.mkdir(parents=True, exist_ok=True)

joblib.dump(final_xgb, model_path)

# Save the parameters used
joblib.dump(best_params, "../models/final_xgboost_params.joblib")

print(f"Model saved to: {model_path.resolve()}")
print("Model parameters saved successfully.")

Training completed in 72.39 seconds
Model saved to: C:\Users\HP\OneDrive\Desktop\HealthCare\models\final_xgboost_model.joblib
Model parameters saved successfully.
